# Unix-terminal. SSH


## Мотивация

SSH — не только «терминал на другом компьютере», а основа удалённой разработки. VS Code Remote SSH, удалённые IDE JetBrains, передача файлов, синхронизация проекта и доступ к Jupyter на сервере так или иначе используют SSH-соединение. Освоив его, можно работать на мощной машине из привычного редактора, переносить данные, оставлять долгие процессы и безопасно открывать закрытые сервисы. Ошибка в настройках SSH легко лишает доступа, поэтому соединение нужно уметь настраивать и проверять самостоятельно.


### ⚠️ Что нужно для этой демки

Часть ячеек ниже ходит на **учебный сервер курса** — это те, где встречается
`course`, `user@host` или `server.example`: разделы про передачу файлов,
`rsync`, туннели и `ssh-keyscan`. На чистой машине без настроенного доступа они
упадут с `Could not resolve hostname`, и это ожидаемо: имя `course` — не
публичный адрес, а алиас, который вы заводите сами.

Прежде чем идти по ноутбуку подряд, добавьте алиас в `~/.ssh/config`,
подставив адрес и логин, выданные вам на курсе:

```
Host course
    HostName ваш-адрес-сервера
    User ваш-логин
    IdentityFile ~/.ssh/id_ed25519
```

и один раз подключитесь `ssh course`, чтобы принять host key. Ячейки, которым
сервер не нужен (генерация ключей, разбор `ssh -G`, локальный `rsync`),
работают и без этого.

## 1. Подключение и удалённая команда

SSH создаёт зашифрованное соединение с удалённой машиной. Адрес имеет форму `user@host`; без пользователя берётся текущее локальное имя.

```bash
# Открыть интерактивную сессию.
ssh user@host

# Выполнить одну команду на сервере и завершить соединение.
ssh user@host whoami
ssh user@host 'uname -a'

# Подключиться к нестандартному порту.
ssh -p 2222 user@host
```

`-p` задаёт порт, `-V` показывает версию клиента. `hostname` печатает имя машины.

`ssh -G host` не подключается к серверу: он печатает итоговые параметры клиента.

Кавычки определяют, где раскроется переменная:

```bash
ssh course "echo $HOME"  # локальная оболочка
ssh course 'echo $HOME'  # удалённая оболочка
```


In [ ]:
%%bash
VALUE=local
bash -c "VALUE=remote; echo \"double quotes: $VALUE\""
bash -c 'VALUE=remote; echo "single quotes: $VALUE"'

mkdir -p ~/seminar-08/config    # сюда складываем всё про настройки SSH-клиента
ssh -V
which ssh
ssh -G student@server.example > ~/seminar-08/config/effective-default.conf 2>/dev/null
head -n 10 ~/seminar-08/config/effective-default.conf


#### ❓ **Вопрос**: Локальный `$HOME` равен `/home/local`, удалённый — `/home/student`. Что выведут `ssh course "echo $HOME"` и `ssh course 'echo $HOME'`?

<details>

<summary><strong>Ответ</strong></summary>

Первая команда передаст уже раскрытый локальный путь `/home/local`. Вторая передаст буквальный `$HOME`, который удалённая оболочка раскроет в `/home/student`.

</details>


## 2. Ключи SSH

Для постоянного доступа используют ключи: пароль можно перебирать через сеть, а закрытый ключ серверу не передаётся. Пара состоит из закрытого и открытого ключа. Закрытый ключ остаётся у владельца; открытый добавляется на сервер в `~/.ssh/authorized_keys`. Парольная фраза защищает закрытый файл при краже.

На Linux закрытый ключ должен быть недоступен другим пользователям. В курсе используем `chmod 400 private_key`: чтение только владельцу. OpenSSH откажется использовать ключ с избыточно широкими правами.

`ssh-keygen` создаёт пару ключей и показывает fingerprint — короткий отпечаток, по которому ключ можно проверить.

- `-t` — тип ключа;
- `-f` — путь;
- `-C` — комментарий;
- `-N` — парольная фраза;
- `-q` — убрать обычные сообщения;
- `-l -f public_key` — показать fingerprint.

Для нового обычного ключа предпочитают `ed25519`: он компактен, быстро создаётся и не требует выбирать размер ключа. RSA не «сломан», но для сопоставимой стойкости ему нужны заметно более длинные ключи; его оставляют для совместимости со старыми системами или требованиями конкретной инфраструктуры. Типы `ed25519-sk` и `ecdsa-sk` работают с аппаратным FIDO-ключом, например YubiKey. DSA устарел и использовать его не следует.

Эта пара ключей нужна для аутентификации — сервер проверяет, что подключается владелец закрытого ключа. Шифрование самого соединения согласуется отдельно, например через ChaCha20-Poly1305 или AES-GCM.

Без конфигурационного файла закрытый ключ выбирают через `ssh -i`:

```bash
chmod 400 ~/seminar-08/keys/course_ed25519
ssh -i ~/seminar-08/keys/course_ed25519 user@host
```

### `ssh-agent`: ключи в текущем окружении

Закрытый ключ с парольной фразой обычно приходится разблокировать при каждом подключении. `ssh-agent` — фоновый процесс, который хранит уже разблокированные ключи в памяти и по запросу выполняет ими аутентификацию. Сам закрытый ключ на сервер не отправляется.

SSH-клиент находит агент по переменной окружения `SSH_AUTH_SOCK`. Поэтому ключи агента доступны текущей оболочке и программам, которые запущены из неё и унаследовали эту переменную. Другая независимая оболочка может быть подключена к другому агенту или не видеть агента совсем.

В графической сессии агент часто запускается автоматически. Сначала проверяют текущую оболочку:

```bash
ssh-add -l
```

Если команда сообщает, что не может подключиться к агенту, его запускают и подключают к текущей оболочке:

```bash
eval "$(ssh-agent -s)"
ssh-add ~/seminar-08/keys/course_ed25519
ssh-add -l
```

`ssh-agent -s` запускает агент и печатает команды, задающие `SSH_AUTH_SOCK` и `SSH_AGENT_PID`. Конструкция `$(...)` получает этот текст, а `eval` выполняет его в текущей оболочке. Простой запуск `ssh-agent` без `eval` лишь напечатает настройки: текущая оболочка не начнёт использовать новый агент. `ssh-add FILE` добавляет закрытый ключ, при необходимости один раз спрашивая парольную фразу; `ssh-add -l` показывает отпечатки добавленных ключей.

Если агент предлагает много ключей, сервер может исчерпать лимит попыток до нужного ключа и разорвать соединение с `Too many authentication failures`. `IdentitiesOnly=yes` заставляет клиент использовать только ключи из `-i` и `IdentityFile`:

```bash
ssh -o IdentitiesOnly=yes -i ~/seminar-08/keys/course_ed25519 user@host
```

`ForwardAgent yes` разрешает удалённой машине обращаться к локальному агенту, не копируя на неё закрытый ключ. Включать это следует только для доверенных промежуточных серверов: получивший доступ к соединению с агентом не прочитает ключ, но сможет временно выполнять им аутентификацию.


In [ ]:
%%bash
mkdir -p ~/seminar-08/keys                  # рабочий каталог демонстрации ключей
rm -f ~/seminar-08/keys/course_ed25519 ~/seminar-08/keys/course_ed25519.pub   # чистый старт: ssh-keygen не перезаписывает молча
ssh-keygen -q -t ed25519 -N '' -C 'course-demo' \
  -f ~/seminar-08/keys/course_ed25519           # -N '' — без парольной фразы, иначе демо спросит ввод
chmod 400 ~/seminar-08/keys/course_ed25519      # закрытый ключ: чтение только владельцу

ls -l ~/seminar-08/keys/course_ed25519*             # два файла: закрытый и парный .pub
ssh-keygen -lf ~/seminar-08/keys/course_ed25519.pub # fingerprint — по нему ключ сверяют глазами
cat ~/seminar-08/keys/course_ed25519.pub            # именно эту строку кладут в authorized_keys


#### ❓ **Вопрос**: Какой файл можно передать на сервер? Ключ добавили в `ssh-agent`, запущенный вручную из одного терминала. Почему в другом терминале этот ключ может быть недоступен?

<details>

<summary><strong>Ответ</strong></summary>

Передают только `course_ed25519.pub`; закрытый `course_ed25519` остаётся на доверенной машине и в курсе получает права `400`. SSH находит агент по `SSH_AUTH_SOCK`. Переменную получают текущая оболочка и её дочерние процессы, а независимый терминал может использовать другое окружение и другой агент.

</details>


## 3. SSH-config


`~/.ssh/config` хранит параметры подключений:

- `Host` — короткое имя;
- `HostName` — адрес;
- `User` — пользователь;
- `Port` — порт;
- `IdentityFile` — закрытый ключ;
- `IdentitiesOnly yes` — не предлагать серверу остальные ключи агента.

`ssh -F file` использует указанный файл настроек вместо стандартного. `ssh -G alias` показывает итоговые параметры до подключения.


In [ ]:
%%bash
# Host course — короткое имя, за которым прячутся адрес, пользователь, порт и ключ
cat > ~/seminar-08/config/ssh_config <<'EOF'
Host course
    HostName server.example
    User student
    Port 2222
    IdentityFile ~/seminar-08/keys/course_ed25519
    IdentitiesOnly yes
EOF


Файл написан. Проверим, во что он разворачивается для имени `course`:
`ssh -G` показывает итоговые параметры и никуда не подключается.


In [ ]:
%%bash
chmod 600 ~/seminar-08/config/ssh_config    # настройки подключения не должны быть доступны другим

ssh -F ~/seminar-08/config/ssh_config -G course \
  > ~/seminar-08/config/effective-course.conf 2>/dev/null   # -F берёт указанный файл вместо ~/.ssh/config
head -n 10 ~/seminar-08/config/effective-course.conf        # hostname, user, port — собраны из Host course


#### ❓ **Вопрос**: Как подключиться с ключом `course_ed25519` без файла настроек и какое поле укажет тот же ключ в `~/.ssh/config`?

<details>

<summary><strong>Ответ</strong></summary>

Без файла настроек: `ssh -i ~/seminar-08/keys/course_ed25519 user@host`. В `~/.ssh/config` используется `IdentityFile ~/seminar-08/keys/course_ed25519`.

</details>


## 4. Проверка сервера и диагностика


### Проверка соединения

`ssh -Tvvv user@host true` проверяет соединение и аутентификацию без интерактивного терминала: `-T` отключает псевдотерминал, `-vvv` включает максимальную клиентскую диагностику, `true` сразу завершает удалённую команду.

`-o Name=Value` передаёт одну настройку клиента. `ConnectTimeout=5` ограничивает установку соединения.


In [ ]:
%%bash
mkdir -p ~/seminar-08/diagnostics
ssh -Tvvv -o ConnectTimeout=5 user@host true \
  > ~/seminar-08/diagnostics/connection.out 2> ~/seminar-08/diagnostics/connection-debug.txt
echo "$?" > ~/seminar-08/diagnostics/connection-exit-code.txt


### Ключ сервера и `known_hosts`

Пользовательский ключ доказывает серверу, кто подключается. Серверный ключ, или host key, решает обратную задачу: помогает клиенту убедиться, что перед ним нужный сервер. При первом подключении SSH показывает fingerprint сервера, а после подтверждения сохраняет его ключ в `~/.ssh/known_hosts`.

Неожиданная смена ключа может означать переустановку сервера или атаку. Новый fingerprint сначала проверяют по независимому доверенному каналу. Нельзя просто удалить старую запись и согласиться с новой.

Серверные ключи должны быть уникальны. При клонировании машины вместе с ключами новый сервер получает чужую идентичность, и различить две машины по ключу становится невозможно. Рассчитывать, что клиент это заметит, не стоит: `known_hosts` просматривается **по имени и адресу текущего соединения**, а не сканируется на повторяющиеся ключи, — к новому имени клиент отнесётся как к незнакомому серверу и просто предложит принять ключ. Поэтому уникальность host key — обязанность того, кто разворачивает сервер: после клонирования образа ключи перегенерируют.

`ssh-keyscan host` получает публичный ключ сервера, но **не подтверждает его подлинность**: он берёт то, что ответила сеть. Дописывать его вывод прямо в `known_hosts`, как в ячейке ниже, допустимо только в учебной демке — так вы записываете «ключ того, кто ответил», а не «ключ нужного сервера». В работе fingerprint сначала сверяют по независимому каналу: с выводом `ssh-keygen -lf` на самом сервере, из панели облака или из письма администратора.

`ssh-keygen` здесь не создаёт новый ключ: `ssh-keygen -F host -f file` ищет запись хоста в выбранном `known_hosts`, `ssh-keygen -R host -f file` удаляет её.


In [ ]:
%%bash
mkdir -p ~/seminar-08/hostkeys
touch ~/seminar-08/hostkeys/known_hosts

# server.example не резолвится (домен .example зарезервирован) — вывод будет пустым;
# на учебном сервере подставьте его имя и увидите строку с ключом
ssh-keyscan server.example \
  >> ~/seminar-08/hostkeys/known_hosts 2>/dev/null || true
cut -c1-70 ~/seminar-08/hostkeys/known_hosts   # что записалось: имя хоста, тип ключа, сам ключ
ssh-keygen -F server.example \
  -f ~/seminar-08/hostkeys/known_hosts || true   # -F ищет запись по имени хоста

#### ❓ **Вопрос**: SSH сообщает, что ключ знакомого сервера изменился. Почему нельзя сразу удалить старую запись?

<details>

<summary><strong>Ответ</strong></summary>

Изменение может означать подмену сервера. Новый fingerprint сначала подтверждают по независимому доверенному каналу.

</details>


## 5. Передача файлов через `scp`

`scp` копирует файлы через SSH и использует те же ключи и настройки подключения. Удалённый путь записывается как `user@host:path`.

```bash
scp report.txt course:reports/
scp course:reports/report.txt returned.txt
```

Путь без начального `/` считается относительно домашнего каталога удалённого пользователя. `/var/tmp/report.txt` начинается от корня удалённой системы.

`-r` копирует каталог, `-P 2222` задаёт SSH-порт. Здесь используется заглавная `P`: строчная `-p` имеет другое значение.

После передачи туда и обратно размер сравнивают через `wc -c`. `cmp first second` ничего не печатает и возвращает `0`, если содержимое файлов совпадает; при различии возвращает ненулевой код.


In [ ]:
%%bash
HOST=course
mkdir -p ~/seminar-08/transfer
echo 'course report' > ~/seminar-08/transfer/report.txt   # файл, который повезём на сервер

ssh "$HOST" 'mkdir -p reports'                            # путь без / — от домашнего каталога на сервере
scp ~/seminar-08/transfer/report.txt "$HOST:reports/"     # локальный путь → удалённый


Файл уехал. Теперь заберём его обратно под другим именем и проверим, что
по дороге ничего не изменилось.


In [ ]:
%%bash
HOST=course
scp "$HOST:reports/report.txt" ~/seminar-08/transfer/returned.txt   # удалённый путь → локальный

wc -c ~/seminar-08/transfer/report.txt ~/seminar-08/transfer/returned.txt   # размеры должны совпасть
cmp ~/seminar-08/transfer/report.txt ~/seminar-08/transfer/returned.txt     # молчит и возвращает 0 = файлы идентичны


#### ❓ **Вопрос**: Куда попадут файлы из `scp file.txt course:reports/` и `scp file.txt course:/reports/`?

<details>

<summary><strong>Ответ</strong></summary>

Первый — в `reports` внутри домашнего каталога удалённого пользователя. Второй — в `/reports` от корня удалённой системы.

</details>


## 6. Синхронизация через `rsync`

`rsync` сравнивает источник и назначение и передаёт изменения:

```bash
rsync -av data/ course:backup/
```

В записи `course:backup/` локальный `rsync` подключается к `course` по SSH и запускает там второй `rsync`. Отдельный сервер rsync настраивать не нужно, но программа должна быть установлена на обеих машинах. Другой SSH-порт задают через `-e 'ssh -p 2222'`.

`scp` удобен для разовой простой копии. `rsync` удобнее для повторной синхронизации дерева: он сравнивает состояния, передаёт изменения и умеет показывать план, исключать пути и удалять лишнее в назначении.

`-a` сохраняет структуру и метаданные, `-v` показывает действия. `data/` означает содержимое каталога, `data` — сам каталог вместе с именем.

`--dry-run` или `-n` показывает план без изменений. `-i` подробно перечисляет изменения. `--delete` удаляет в назначении то, чего нет в источнике. `--exclude='pattern'` исключает совпавшие пути. Перед удалением всегда проверяют dry-run.


In [ ]:
%%bash
mkdir -p ~/seminar-08/rsync/source ~/seminar-08/rsync/copy
echo alpha > ~/seminar-08/rsync/source/a.txt
echo beta > ~/seminar-08/rsync/source/b.txt

# -n показывает только план, ничего не меняя; -i расшифровывает каждое действие
rsync -avni --delete --exclude='__pycache__/' \
  ~/seminar-08/rsync/source/ ~/seminar-08/rsync/copy/


План устраивает — лишнего удаления в нём нет. Повторяем ту же команду без
`-n`: теперь изменения действительно применяются.


In [ ]:
%%bash
rsync -avi --delete --exclude='__pycache__/' \
  ~/seminar-08/rsync/source/ ~/seminar-08/rsync/copy/

ls -l ~/seminar-08/rsync/copy    # в назначении появились оба файла


#### ❓ **Вопрос**: Когда для каталога разумнее выбрать `rsync`, а не `scp`, и чем отличаются источники `data` и `data/`?

<details>

<summary><strong>Ответ</strong></summary>

`rsync` выбирают для повторной синхронизации: он сравнит состояния и передаст изменения. `scp` подходит для простой разовой копии. `data` копирует сам каталог с его именем, `data/` — только его содержимое.

</details>


## 7. `tmux`: работа после отключения


Обычная удалённая команда и процессы, привязанные к SSH-терминалу, при разрыве соединения теряют терминал и обычно завершаются. Простой запуск через `&` не гарантирует, что процесс переживёт отключение.

`tmux` запускает на сервере собственную сессию с терминалами. Она продолжает работать и после `Ctrl+B`, затем `D`, и после внезапного разрыва SSH без явного отсоединения.

```bash
# Создать интерактивную сессию.
tmux new -s work

# Вернуться к существующей сессии.
tmux attach -t work
```

Комбинации внутри tmux начинаются с префикса: нажмите `Ctrl+B`, отпустите, затем нажмите следующую клавишу. В краткой записи `C-b` означает `Ctrl+B`.

- `C-b d` — отсоединиться;
- `C-b c` — создать окно;
- `C-b n`, `C-b p`, `C-b 0…9` — перейти между окнами;
- `C-b %` — разделить окно на левую и правую панели;
- `C-b "` — разделить окно на верхнюю и нижнюю панели;
- `C-b` и стрелка — перейти в соседнюю панель;
- `C-b q` — показать номера панелей.

Для запуска без подключения к экрану:

```bash
tmux new-session -d -s worker 'sleep 300'
tmux list-sessions
tmux list-panes -t worker -F '#{pane_pid} #{pane_current_command}'
tmux has-session -t worker
tmux kill-session -t worker
```

`-d` создаёт отсоединённую сессию, `-s` задаёт имя, `-t` выбирает существующую сессию, `-F` задаёт формат вывода. `#{pane_pid}` и `#{pane_current_command}` — имена полей tmux. `has-session` возвращает код `0`, если указанная сессия существует.

`nohup COMMAND > run.log 2>&1 &` — более простой способ отвязать неинтерактивную команду от терминала, но к её экрану нельзя вернуться. `tmux` подходит для интерактивных экспериментов и наблюдения. Долго работающий сервис не держат в tmux: им управляет systemd с политикой запуска, перезапуска и журналами. Деплой также должен быть воспроизводимым скриптом или автоматизированным процессом, а не действиями в сохранённом терминале.


#### ❓ **Вопрос**: SSH-соединение оборвалось, пока пользователь работал внутри `tmux` и не нажимал detach. Завершится ли tmux-сессия и как к ней вернуться?

<details>

<summary><strong>Ответ</strong></summary>

Сессия продолжит работать на сервере. После нового подключения к серверу к ней возвращаются через `tmux attach -t NAME`.

</details>


## Дополнительно


### Установка публичного ключа

Первичный доступ обычно выдаёт администратор или облачная платформа. Пока этот доступ действует, публичный ключ можно одной командой добавить в `~/.ssh/authorized_keys`:

```bash
ssh-copy-id -i ~/seminar-08/keys/course_ed25519.pub course
```

`ssh-copy-id` подключается по SSH, создаёт нужные файлы и не добавляет уже установленный ключ повторно. Флаг `-i` выбирает публичный ключ. Новый вход по ключу проверяют во втором терминале, не закрывая рабочее или аварийное соединение. Для постоянного доступа к серверу используют ключи, а парольную аутентификацию отключают только после такой проверки.


### Туннели

В `-L 9000:127.0.0.1:8888`:

- `9000` — порт на локальной машине;
- `127.0.0.1:8888` — адрес сервиса со стороны SSH-сервера;
- `course` — сервер, через который идёт соединение.

```bash
ssh -N -o ExitOnForwardFailure=yes \
  -L 9000:127.0.0.1:8888 course
```

`-L` создаёт локальный проброс, `-N` не запускает удалённую команду. `ExitOnForwardFailure=yes` завершает SSH, если клиент не смог открыть проброс. Пока команда работает, локальный адрес проверяют обычным клиентом сервиса, например `curl http://127.0.0.1:9000`.


### Окна и вывод `tmux`

Иерархия tmux: **сессия → окна → панели**. В примере сессия называется `demo`, окна — `clock` и `identity`; в каждом окне пока одна панель.

```bash
tmux new-session -d -s demo -n clock 'date; sleep 60'
tmux new-window -t demo -n identity 'whoami; sleep 60'
tmux list-windows -t demo
tmux capture-pane -p -t demo:clock
```

Команды читаются так:

- `new-session -d -s demo -n clock COMMAND` — создать отсоединённую сессию `demo`, назвать первое окно `clock` и запустить в нём `COMMAND`;
- `new-window -t demo -n identity COMMAND` — добавить в выбранную через `-t` сессию окно `identity`;
- `list-windows -t demo` — показать окна этой сессии;
- `capture-pane -p -t demo:clock` — вывести содержимое панели окна `clock` в stdout. В адресе `session:window` двоеточие отделяет имя сессии от имени окна.

`sleep 60` оставляет окно живым после завершения первой команды. Без продолжающегося процесса окно сразу закроется.


### Серверная сторона: `sshd`

`ssh` — клиент, а `sshd` — серверный процесс, который принимает соединения. Его настройки находятся в `/etc/ssh/sshd_config` и дополнительных файлах настроек дистрибутива.

Ошибочная настройка может отрезать удалённый доступ. Перед применением конфигурацию проверяют через `sudo sshd -t`, текущую сессию не закрывают, а новое подключение проверяют во втором терминале. После проверки конфигурацию перечитывает служба `ssh` или `sshd` — имя зависит от дистрибутива. Если все способы входа уже сломаны, нужен аварийный режим облака, последовательная консоль или другой независимый доступ.


### Журналы SSH и `fail2ban`

Открытый в интернет SSH-сервер постоянно получает автоматические попытки входа по популярным именам и паролям. Живой журнал обычно смотрят одной из команд:

```bash
sudo journalctl -u ssh -f
sudo journalctl -u sshd -f
```

Имя службы зависит от дистрибутива. Строки вида `Failed password for invalid user ...` показывают не «целевую атаку», а обычный сетевой шум. Поэтому постоянный вход настраивают по ключам, а не по паролю.

`fail2ban` читает журналы и временно блокирует адреса после серии неудачных попыток. Это дополнительный барьер, а не замена ключам, обновлениям и сетевым ограничениям.
